# Phase C2 Step 1: Pseudo-label Unlabeled Soundscapes
Load all 5 C1 fold models, run sliding-window inference on 10,592 unlabeled train_soundscapes,
and save pseudo-labels as a CSV for use in C2 retraining.

In [8]:
# Cell 1: Config + Imports

import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import librosa
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

CFG = {
    'base_dir': '/data/scratch/scienceteam/jupyter-mir/bird/Data/birdclef-2026',
    'model_dir': '/data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed',
    'output_path': '/data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/pseudo_labels.csv',
    'sr': 32000,
    'duration': 20,
    'n_fft': 4096,
    'hop_length': 1252,
    'n_mels': 224,
    'fmin': 0,
    'fmax': 16000,
    'top_db': 80.0,
    'img_width': 512,
    'model_name': 'tf_efficientnet_b0.ns_jft_in1k',
    'num_classes': 234,
    'n_folds': 5,
    'window_sec': 20,
    'step_sec': 5,
    'batch_size': 512,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

taxonomy = pd.read_csv(os.path.join(CFG['base_dir'], 'taxonomy.csv'))
LABELS = sorted(taxonomy['primary_label'].tolist())

print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Target species: {len(LABELS)}")

Device: cuda (NVIDIA A40)
Target species: 234


In [9]:
# Cell 2: Model definition (identical to C1)

class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=-1).pow(1.0 / self.p)


class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg['model_name'], pretrained=False,
            in_chans=3, num_classes=0, global_pool=''
        )
        self.feat_dim = 1280
        self.freq_pool = GeM(p=3)
        self.fc = nn.Linear(self.feat_dim, cfg['num_classes'])

    def forward(self, x):
        features = self.backbone(x)
        x = self.freq_pool(features)
        x = x.transpose(1, 2)
        frame_logits = self.fc(x)
        clip_logits = frame_logits.mean(dim=1)
        clip_logits_max = frame_logits.max(dim=1)[0]
        output = 0.5 * clip_logits + 0.5 * clip_logits_max
        return output

print("SEDModel defined.")

SEDModel defined.


In [10]:
# Cell 3: Load all 5 fold models

models = []
for fold in range(CFG['n_folds']):
    path = os.path.join(CFG['model_dir'], f'c1_fold{fold}.pth')
    model = SEDModel(CFG).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    models.append(model)
    print(f"Loaded fold {fold}: {path}")

print(f"\n{len(models)} models loaded and ready.")

Loaded fold 0: /data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/c1_fold0.pth
Loaded fold 1: /data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/c1_fold1.pth
Loaded fold 2: /data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/c1_fold2.pth
Loaded fold 3: /data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/c1_fold3.pth
Loaded fold 4: /data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/c1_fold4.pth

5 models loaded and ready.


In [12]:
# Cell 4: Audio + mel spectrogram helpers (same params as C1)

def load_full_audio(filepath):
    """Load entire soundscape at target sample rate."""
    try:
        y, sr = librosa.load(filepath, sr=CFG['sr'])
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return np.zeros(CFG['sr'] * 60, dtype=np.float32)
    return y.astype(np.float32)


def audio_to_melspec(y):
    """Convert raw waveform to normalized mel spectrogram tensor."""
    # Absmax normalization
    max_val = np.abs(y).max()
    if max_val > 0:
        y = y / max_val

    mel = librosa.feature.melspectrogram(
        y=y, sr=CFG['sr'],
        n_fft=CFG['n_fft'],
        hop_length=CFG['hop_length'],
        n_mels=CFG['n_mels'],
        fmin=CFG['fmin'],
        fmax=CFG['fmax'],
    )
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=CFG['top_db'])
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    # Pad/trim to fixed width
    if mel_db.shape[1] < CFG['img_width']:
        mel_db = np.pad(mel_db, ((0, 0), (0, CFG['img_width'] - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :CFG['img_width']]

    mel_3ch = np.stack([mel_db, mel_db, mel_db], axis=0)
    return torch.tensor(mel_3ch, dtype=torch.float32)


print("Helpers defined.")

Helpers defined.


In [13]:
# Batch size test for pseudo-labeling inference
import torch
import gc

# Make sure models are loaded already (Cell 3)
test_mel = audio_to_melspec(np.zeros(CFG['sr'] * CFG['window_sec'], dtype=np.float32))

for bs in [32, 64, 128, 180, 256, 512]:
    try:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        gc.collect()
        
        batch = torch.stack([test_mel] * bs).to(DEVICE)
        with torch.no_grad():
            for model in models:
                _ = model(batch)
        
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"BS {bs}: OK (peak GPU: {peak:.1f} GB / 48 GB)")
        del batch
        torch.cuda.empty_cache()
    except RuntimeError as e:
        print(f"BS {bs}: OOM")
        torch.cuda.empty_cache()
        gc.collect()
        break

BS 32: OK (peak GPU: 1.1 GB / 48 GB)
BS 64: OK (peak GPU: 2.1 GB / 48 GB)
BS 128: OK (peak GPU: 4.2 GB / 48 GB)
BS 180: OK (peak GPU: 5.8 GB / 48 GB)
BS 256: OK (peak GPU: 8.2 GB / 48 GB)
BS 512: OK (peak GPU: 16.4 GB / 48 GB)


In [14]:
# Cell 5: Sliding window inference function

@torch.no_grad()
def predict_soundscape(audio, models):
    """Run sliding-window inference on one soundscape.
    
    Returns a dict: {start_sec: probability_array} for each 5-sec segment.
    Each 5-sec segment gets predictions from up to 4 overlapping 20-sec windows,
    averaged across windows and across all fold models.
    """
    sr = CFG['sr']
    window_samples = CFG['window_sec'] * sr   # 640,000
    step_samples = CFG['step_sec'] * sr        # 160,000
    total_samples = len(audio)
    total_sec = total_samples / sr

    # Grid of 5-sec segments
    segment_starts = list(range(0, int(total_sec) - CFG['step_sec'] + 1, CFG['step_sec']))
    n_segments = len(segment_starts)

    # Accumulator: sum of predictions and count for averaging
    pred_sum = np.zeros((n_segments, CFG['num_classes']), dtype=np.float64)
    pred_count = np.zeros(n_segments, dtype=np.float64)

    # Generate all windows
    windows = []
    window_meta = []  # (window_start_sec, window_end_sec)
    for start_sample in range(0, total_samples - window_samples + 1, step_samples):
        chunk = audio[start_sample:start_sample + window_samples]
        if len(chunk) < window_samples:
            break
        mel = audio_to_melspec(chunk)
        windows.append(mel)
        start_sec = start_sample / sr
        end_sec = start_sec + CFG['window_sec']
        window_meta.append((start_sec, end_sec))

    if len(windows) == 0:
        # File too short -- pad and run single window
        padded = np.zeros(window_samples, dtype=np.float32)
        padded[:len(audio)] = audio
        mel = audio_to_melspec(padded)
        windows.append(mel)
        window_meta.append((0, CFG['window_sec']))

    # Batch inference across all windows
    all_preds = np.zeros((len(windows), CFG['num_classes']), dtype=np.float64)
    batch_size = CFG['batch_size']

    for i in range(0, len(windows), batch_size):
        batch = torch.stack(windows[i:i + batch_size]).to(DEVICE)
        batch_pred = np.zeros((batch.shape[0], CFG['num_classes']), dtype=np.float64)

        for model in models:
            logits = model(batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            batch_pred += probs

        batch_pred /= len(models)  # Average across folds
        all_preds[i:i + batch.shape[0]] = batch_pred

    # Map window predictions to 5-sec segments
    for w_idx, (w_start, w_end) in enumerate(window_meta):
        for s_idx, seg_start in enumerate(segment_starts):
            seg_end = seg_start + CFG['step_sec']
            if seg_start >= w_start and seg_end <= w_end:
                pred_sum[s_idx] += all_preds[w_idx]
                pred_count[s_idx] += 1

    # Average overlapping predictions
    pred_count = np.maximum(pred_count, 1)  # avoid division by zero
    predictions = pred_sum / pred_count[:, None]

    return {seg_start: predictions[i] for i, seg_start in enumerate(segment_starts)}


print("Inference function defined.")

Inference function defined.


In [15]:
# Cell 6: Main loop -- pseudo-label all unlabeled soundscapes

# Identify labeled soundscapes to exclude
labels_df = pd.read_csv(os.path.join(CFG['base_dir'], 'train_soundscapes_labels.csv'))
labeled_files = set(labels_df['filename'].unique())
print(f"Labeled soundscapes to exclude: {len(labeled_files)}")

# All soundscape files
soundscape_dir = os.path.join(CFG['base_dir'], 'train_soundscapes')
all_files = sorted(glob.glob(os.path.join(soundscape_dir, '*.ogg')))
print(f"Total soundscapes: {len(all_files)}")

# Filter to unlabeled only
unlabeled_files = [f for f in all_files if os.path.basename(f) not in labeled_files]
print(f"Unlabeled soundscapes to pseudo-label: {len(unlabeled_files)}")

# Run inference
rows = []
for filepath in tqdm(unlabeled_files, desc="Pseudo-labeling"):
    soundscape_id = os.path.splitext(os.path.basename(filepath))[0]
    audio = load_full_audio(filepath)
    preds = predict_soundscape(audio, models)

    for start_sec, probs in preds.items():
        row = {'soundscape_id': soundscape_id, 'start_sec': int(start_sec)}
        for label, prob in zip(LABELS, probs):
            row[label] = float(prob)
        rows.append(row)

print(f"\nTotal pseudo-label rows: {len(rows):,}")

Labeled soundscapes to exclude: 66
Total soundscapes: 10658
Unlabeled soundscapes to pseudo-label: 10592


Pseudo-labeling: 100%|██████████| 10592/10592 [52:56<00:00,  3.33it/s]


Total pseudo-label rows: 127,104


In [16]:
# Cell 7: Save pseudo-labels

pseudo_df = pd.DataFrame(rows)
pseudo_df.to_csv(CFG['output_path'], index=False)

print(f"Saved to: {CFG['output_path']}")
print(f"Shape: {pseudo_df.shape}")
print(f"Unique soundscapes: {pseudo_df['soundscape_id'].nunique()}")
print(f"\nSample predictions (first 3 rows, top 5 species):")
species_cols = LABELS
for i in range(min(3, len(pseudo_df))):
    row = pseudo_df.iloc[i]
    top5 = row[species_cols].astype(float).nlargest(5)
    print(f"\n  {row['soundscape_id']} @ {int(row['start_sec'])}s:")
    for label, prob in top5.items():
        print(f"    {label}: {prob:.4f}")

print("\nDone! Ready for C2 retraining.")

Saved to: /data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/pseudo_labels.csv
Shape: (127104, 236)
Unique soundscapes: 10592

Sample predictions (first 3 rows, top 5 species):

  BC2026_Train_0067_S04_20240810_000000 @ 0s:
    ruther1: 0.9829
    brnowl: 0.9799
    burowl: 0.9770
    555146: 0.9731
    linwoo1: 0.9689

  BC2026_Train_0067_S04_20240810_000000 @ 5s:
    ruther1: 0.9858
    brnowl: 0.9856
    burowl: 0.9722
    linwoo1: 0.9693
    y00678: 0.9693

  BC2026_Train_0067_S04_20240810_000000 @ 10s:
    brnowl: 0.9856
    strowl1: 0.9280
    compot1: 0.9258
    trsowl: 0.9172
    linwoo1: 0.9008

Done! Ready for C2 retraining.
